# 第 4 章 · 第一个量化策略：双均线回测（零安装体验版）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xlzxld/LH/blob/main/notebooks/ch04-dual-ma-quickstart.ipynb)

> **这是给「还没装好环境」的你准备的**：在浏览器里点几下就能跑完人生第一次量化回测——
> 不用安装 Python、不用下载数据、不用联网调接口。

**三步开始**

1. 菜单 `代码执行程序 → 全部运行`（Colab）/ `Run All`（本地 Jupyter）
2. 从上往下看每一步的输出
3. 做完最后的 4 道动手练习

**配套正式教程**：[docs/04-第一个策略-双均线回测.md](../docs/04-第一个策略-双均线回测.md) ｜ **目录**：[README](../README.md)

> 本笔记本自带一份「固定种子」的模拟行情，所以**每次运行结果完全一致**，
> 你可以放心对照这里的预期输出。想跑真实行情（沪深300ETF 等），见文末「下一步」。

## 你将学到

| 步骤 | 做什么 | 关键概念 |
|---|---|---|
| 1 | 造一份可复现的行情数据 | 固定随机种子、OHLCV |
| 2 | 把「金叉买入、死叉卖出」写成代码 | 移动平均线、目标仓位、NaN 语义 |
| 3 | 用四个规矩跑一次回测 | 次根开盘成交、手续费、滑点、无未来函数 |
| 4 | 看懂结果并诚实面对它 | 年化收益、最大回撤、夏普比率、买入持有基准 |

> ⚠️ **一句话忠告**：回测赚钱 ≠ 实盘赚钱。本策略在震荡市大概率跑不赢「买入持有」——
> 如实面对回测结果，正是这个教程要教你的第一课。

In [ ]:
# ---- 第 0 步：检查环境（Colab 已自带这三个库，无需安装）----
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python     :", sys.version.split()[0])
print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("matplotlib :", plt.matplotlib.__version__)
print("\n环境就绪，开始。")

## 第 1 步 · 造一份「固定种子」的行情数据

回测要有价格数据。真实数据要联网、还会被限流，所以我们先用**模拟数据**把流程跑通。

关键在「固定种子」（`seed=42`）：随机数一旦被固定，**任何人、任何电脑、任何时候**
生成的数列都一模一样。这样教程里写的「预期输出」你才能逐位对上。

K 线数据有五个字段（量化黑话叫 OHLCV）：

| 字段 | 含义 |
|---|---|
| `open` | 当日开盘价 |
| `high` | 当日最高价 |
| `low` | 当日最低价 |
| `close` | 当日收盘价 |
| `volume` | 成交量 |

> 我们**只用收盘价算信号、用次日开盘价成交**——这是「没有偷看未来」的关键，第 3 步细说。

In [ ]:
def make_sample_ohlcv(rows=250, seed=42):
    """固定种子生成 OHLCV 日线：几何随机游走（日收益均值 0.05%、波动 1.5%）。"""
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range("2023-01-02", periods=rows)        # 工作日日线 → A股口径一年 252 根
    close = 100.0 * np.cumprod(1.0 + rng.normal(0.0005, 0.015, rows))
    open_ = close * (1.0 + rng.normal(0.0, 0.003, rows))
    body_hi = np.maximum(open_, close)
    body_lo = np.minimum(open_, close)
    spread = np.abs(rng.normal(0.0, 0.004, rows)) + 1e-4    # 影线幅度（恒正）
    df = pd.DataFrame({
        "open": open_,
        "close": close,
        "high": body_hi * (1.0 + spread),
        "low": body_lo * (1.0 - spread),
        "volume": rng.integers(8_000, 20_000, rows).astype(float),
    }, index=idx)
    df.index.name = "date"
    return df


df = make_sample_ohlcv()   # 与 code/ch03_data/make_sample_data.py 完全同一份数据
print(f"数据范围: {df.index[0].date()} ~ {df.index[-1].date()}，共 {len(df)} 根K线")
print(f"收盘价  : {df['close'].iloc[0]:.2f} -> {df['close'].iloc[-1]:.2f}"
      f"（区间涨跌 {df['close'].iloc[-1] / df['close'].iloc[0] - 1:+.2%}）")
df.head()

数据合法吗？一条铁律：

$$\text{low} \le \min(\text{open},\ \text{close}) \le \max(\text{open},\ \text{close}) \le \text{high}$$

下面顺手自检一遍——真实数据里也常有脏数据，**跑策略之前先体检是好习惯**。

In [ ]:
# ---- 自检：OHLC 关系是否合法（真实数据同样必须过这一关）----
bad = ((df["low"] > df[["open", "close"]].min(axis=1))
       | (df["high"] < df[["open", "close"]].max(axis=1)))
print(f"非法K线行数: {int(bad.sum())}（应为 0）")

# ---- 看一眼价格走势 ----
# 图例故意用英文：Colab 默认没有中文字体，中文会显示成方框（解法见 docs/13 FAQ）
ax = df["close"].plot(figsize=(10, 3.5), color="#c0392b", lw=1.2)
ax.set_title("Sample close price (fixed seed=42)", fontsize=11)
ax.set_ylabel("Price")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 第 2 步 · 把「金叉买入、死叉卖出」写成代码

**双均线策略**（最经典、最容易懂的择时策略）：

- 算两条均线：**快线**（近 20 日均价）、**慢线**（近 60 日均价）
- 快线**上穿**慢线 → 「金叉」→ 买入（满仓）
- 快线**下穿**慢线 → 「死叉」→ 卖出（空仓）

直觉是：快线反应灵敏、慢线代表趋势；快线跑到慢线上面，说明近期在走强。

**关键设计**：策略不直接输出「买 / 卖」，而是输出**目标仓位**（`0` = 空仓，`1` = 满仓）。
这样「回测」和「实盘」就能共用同一份信号代码，语义不会各说各话。

> **最容易踩的坑**：前 60 天数据不够，慢线还没算出来，值是 `NaN`。
> `NaN` 的语义是「**数据不足，维持现状不动**」——**不是「清仓」**！
> 写错的话，回测会凭空多出一堆假的买卖，结果全废。

In [ ]:
def dual_ma_weights(df, fast=20, slow=60):
    """双均线信号 -> 目标仓位序列。NaN 表示「数据不足，维持现状不动」。"""
    ma_fast = df["close"].rolling(fast).mean()
    ma_slow = df["close"].rolling(slow).mean()
    weights = pd.Series(float("nan"), index=df.index)
    weights[(ma_fast > ma_slow) & ma_slow.notna()] = 1.0     # 金叉状态：满仓
    weights[(ma_fast <= ma_slow) & ma_slow.notna()] = 0.0    # 死叉状态：空仓
    return weights


weights = dual_ma_weights(df, fast=20, slow=60)
print(f"NaN（数据不足）根数: {int(weights.isna().sum())}（前 59 根，慢线尚未就绪）")
print(f"满仓根数: {int((weights == 1.0).sum())}，空仓根数: {int((weights == 0.0).sum())}")

flips = weights.dropna().diff().dropna()
print(f"金叉次数: {int((flips > 0).sum())}，死叉次数: {int((flips < 0).sum())}")
weights.tail(8)

## 第 3 步 · 回测：四个必须守住的规矩

新手回测结果「好得不像真的」，99% 是因为破了下面某一条：

| 规矩 | 怎么做 | 破了会怎样 |
|---|---|---|
| ① 不许偷看未来 | 用**今天收盘**算信号，**明天开盘**才成交 | 拿当天收盘价成交＝开了上帝视角，收益虚高 |
| ② 手续费要算 | 每笔按成交额收佣金 | 交易越频繁，被吃掉得越多 |
| ③ 滑点要算 | 买入成交价 = 理论价 × (1 + 滑点) | 实盘成交价比你以为的差一点 |
| ④ 别无视碎单 | 调仓金额太小就跳过 | 为几十块钱反复买卖，白交手续费 |

下面这个「迷你回测引擎」只有 30 行，但四条规矩全都落实了。
（正式版在 [code/ch04_backtest/engine.py](../code/ch04_backtest/engine.py)，还支持止损、印花税等。）

In [ ]:
def run_backtest_mini(df, weights, initial_cash=100_000.0,
                      commission_rate=2.5e-4,   # 单边手续费 0.025%
                      slippage_rate=5e-4,       # 单边滑点 0.05%
                      min_trade_pct=0.005):     # 调仓额小于总权益 0.5% 就跳过
    """迷你回测引擎：无未来函数（次根开盘成交）+ 手续费 + 滑点 + 碎单过滤。"""
    cash, shares = float(initial_cash), 0.0
    pending, trades, equity = None, [], []

    for i, (when, row) in enumerate(df.iterrows()):
        open_px, close_px = float(row["open"]), float(row["close"])

        # ① 先执行「昨天收盘产生、今天开盘成交」的挂单 —— 无未来函数的核心
        if i > 0 and pending is not None:
            equity_now = cash + shares * open_px
            delta = pending * equity_now - shares * open_px   # >0 要买、<0 要卖
            if abs(delta) >= min_trade_pct * equity_now:      # ④ 碎单过滤
                if delta > 0:                                 # ---- 买入
                    buy_px = open_px * (1.0 + slippage_rate)  # ③ 滑点：买得贵一点
                    notional = min(delta, cash / (1.0 + commission_rate))
                    if notional > 0:
                        fee = notional * commission_rate      # ② 手续费
                        shares += notional / buy_px
                        cash -= notional + fee
                        trades.append({"date": when.date(), "side": "BUY",
                                       "price": round(buy_px, 4),
                                       "shares": round(notional / buy_px, 4),
                                       "fee": round(fee, 2)})
                else:                                        # ---- 卖出
                    sell_px = open_px * (1.0 - slippage_rate) # ③ 滑点：卖得便宜一点
                    sold = min(-delta / sell_px, shares)      # 不做空：只能卖手里的
                    if sold > 0:
                        notional = sold * sell_px
                        fee = notional * commission_rate      # ② 手续费
                        cash += notional - fee
                        shares -= sold
                        trades.append({"date": when.date(), "side": "SELL",
                                       "price": round(sell_px, 4),
                                       "shares": round(sold, 4),
                                       "fee": round(fee, 2)})
            pending = None

        equity.append(cash + shares * close_px)               # 收盘记净值

        w = weights.iloc[i]                                   # 收盘算新信号
        pending = None if pd.isna(w) else float(w)             # 排到下一根开盘执行

    return pd.Series(equity, index=df.index, name="equity"), trades


equity, trades = run_backtest_mini(df, weights)
print(f"成交 {len(trades)} 笔：\n")
for t in trades:
    print(f"  {t['date']}  {t['side']:<4}  价格 {t['price']:>8.3f}"
          f"  份额 {t['shares']:>9.3f}  手续费 {t['fee']:>6.2f}")

## 第 4 步 · 看懂结果（比跑通更重要）

净值曲线 = 账户总资产随时间的变化。但光看一条线不够，要算成指标才好比较：

| 指标 | 人话解释 | 关注点 |
|---|---|---|
| 总收益率 | 从头到尾赚了百分之几 | 直观，但没考虑时间长短 |
| 年化收益率 (CAGR) | 折算成「一年赚多少」 | 3 个月赚 50% ≠ 厉害，看年化 |
| 年化波动率 | 净值上下颠簸的剧烈程度 | 越大越睡不着觉 |
| 夏普比率 | 每承受 1 份波动，换来多少收益 | >1 不错，>2 优秀，<0 不如存银行 |
| **最大回撤** | 从最高点最多跌了多少 | **最重要**——决定你拿不拿得住 |
| 交易次数 | 一共买卖了几次 | 太少说明结果可能只是运气 |

In [ ]:
def compute_stats(equity, periods_per_year=252, risk_free_rate=0.02):
    """从净值曲线算绩效指标（口径与 code/ch05_metrics/performance.py 一致）。"""
    equity = equity.dropna()
    returns = equity.pct_change().dropna()
    total_return = equity.iloc[-1] / equity.iloc[0] - 1.0
    years = (len(equity) - 1) / periods_per_year                 # 年化：按 252 个交易日
    cagr = (1.0 + total_return) ** (1.0 / years) - 1.0 if years > 0 else 0.0
    vol = returns.std(ddof=1) * np.sqrt(periods_per_year)
    excess = returns.mean() * periods_per_year - risk_free_rate  # 夏普要减无风险利率
    sharpe = excess / vol if vol > 1e-12 else 0.0
    mdd = float((equity / equity.cummax() - 1.0).min())          # 最大回撤
    return {"总收益率": total_return, "年化收益率(CAGR)": cagr,
            "年化波动率": vol, "夏普比率": sharpe, "最大回撤": mdd}


benchmark = equity.iloc[0] * df["close"] / df["close"].iloc[0]  # 买入持有基准
stats = compute_stats(equity)
bench = compute_stats(benchmark)
total_fee = sum(t["fee"] for t in trades)

pct_keys = ["总收益率", "年化收益率(CAGR)", "年化波动率", "最大回撤"]
summary = pd.DataFrame({
    "双均线策略": [f"{stats[k]:.2%}" for k in pct_keys]
                  + [f"{stats['夏普比率']:.2f}", str(len(trades)), f"{total_fee:.2f}"],
    "买入持有": [f"{bench[k]:.2%}" for k in pct_keys]
                + [f"{bench['夏普比率']:.2f}", "-", "-"],
}, index=pct_keys + ["夏普比率", "交易次数", "累计手续费(元)"])

print(f"结论速览：策略 {stats['总收益率']:+.2%} vs 买入持有 {bench['总收益率']:+.2%}")
summary

In [ ]:
# ---- 净值曲线对比：策略 vs 买入持有（都以 1.0 起步才可比）----
ax = (equity / equity.iloc[0]).plot(figsize=(10, 4), color="#c0392b",
                                    lw=1.6, label="Dual MA strategy")
(benchmark / benchmark.iloc[0]).plot(ax=ax, color="#7f8c8d", lw=1.2,
                                     ls="--", label="Buy & Hold")
ax.set_title("Equity curve: Dual MA vs Buy & Hold (sample data)", fontsize=11)
ax.set_ylabel("Net value (start = 1.0)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 诚实清单：这个结果说明了什么？

先看真实输出：策略 **-7.60%** vs 买入持有 **-8.35%**，6 笔交易共付手续费 **148.13 元**。

- **两个都是亏的**。这段模拟行情整体下跌 8.35%，空仓策略「少亏一点」不等于「会赚钱」——
  择时策略的价值主要在下跌市控制回撤，而不是无中生有地盈利。
- 这是**模拟数据、单一样本**，只能证明「流程跑通了」，不能据此判断策略好坏。
- **单次回测的结论非常脆弱**：换个随机种子（练习 4）结论就可能翻转——
  这正是「参数优化」与「样本外验证」存在的意义（第 7 章）。
- **手续费是真实成本**：6 笔就交了 148 元；换成 0.2% 费率的市场（练习 3）会被吃得更狠。

## 4 道动手练习

1. **改参数**：把 `dual_ma_weights(df, fast=20, slow=60)` 改成 `fast=5, slow=20`，重跑第 2~4 步。
   快线更灵敏了，交易次数和累计手续费怎么变？
2. **体验「未来函数」**：把迷你引擎里的成交价从 `row["open"]` 改成 `row["close"]`
   （当天收盘就成交）。结果变好看还是变差？为什么这种「好看」是假的？
3. **加成本**：把 `commission_rate` 从 `2.5e-4` 改成 `2e-3`（0.2%，接近币安现货），结果差多少？
4. **换数据**：把 `seed=42` 改成别的数字（如 7、2024），多跑几次。
   结论始终是「跑不赢买入持有」吗？这说明单次回测的结论有多脆弱？

> 参考答案与讲解见 [solutions/ch04.md](../solutions/ch04.md)。

## 下一步：装上环境，用真实数据跑

笔记本用的是模拟数据。要走完完整教程（真实 A 股 / 币安数据、绩效进阶、参数优化、实盘），
回仓库按下面几步来：

```bash
python -m venv .venv
# Windows PowerShell（Mac/Linux: source .venv/bin/activate）
.venv\Scripts\Activate.ps1
pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple
python scripts/setup_check.py                 # 环境自检：哪里有问题一眼看出来
python code/ch04_backtest/run_dual_ma.py      # 真实数据跑双均线（沪深300ETF）
```

- **完全离线跑通**：`python code/ch03_data/make_sample_data.py` 生成与本文同一份种子数据，
  再 `python code/ch04_backtest/run_dual_ma.py --csv data/sample_prices.csv`
- **一键验证全链路**（数据 → 单测 → 回测 → 产物）：`python scripts/verify_all.py`
- **看不懂命令行报错** → [第 0 章 环境搭建](../docs/00-学习路线图与环境搭建.md)
- **卡住了** → [第 13 章 常见问题 FAQ](../docs/13-常见问题FAQ与进阶路线.md)

> ⚠️ 本项目仅用于学习研究，不构成投资建议。回测赚钱不等于实盘赚钱。